# deep-ts-imputer — Quickstart

This notebook walks through the full pipeline on the synthetic Seine-like dataset:

1. Generate the dataset
2. Load and split it chronologically
3. Build a sliding-window supervised dataset
4. Train a CNN-BiLSTM model
5. Evaluate it on the held-out test set
6. Reconstruct the missing values in the original CSV

Run it after `pip install -e .` from the repo root.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from deep_ts_imputer.utils.config import load_config
from deep_ts_imputer.utils.seed import set_global_seed
from deep_ts_imputer.data.dataset import load_timeseries, select_features
from deep_ts_imputer.data.preprocessing import chronological_split
from deep_ts_imputer.data.windowing import sliding_window
from deep_ts_imputer.models.factory import build_model, available_models
from deep_ts_imputer.training.trainer import fit_model
from deep_ts_imputer.evaluation.metrics import compute_all
from deep_ts_imputer.imputation.reconstructor import reconstruct_series

print('Available models:', available_models())

In [ ]:
# 1. Generate synthetic data (skip if data/synthetic.csv already exists)
!python ../scripts/generate_synthetic_demo.py --out ../data/synthetic.csv

In [ ]:
# 2. Load config and dataset
cfg = load_config('../configs/synthetic_demo.yaml')
set_global_seed(cfg.seed)
df = load_timeseries(cfg.data.path, date_column=cfg.data.date_column,
                     date_begin=cfg.data.date_begin, date_end=cfg.data.date_end)
inputs, targets = select_features(df, cfg.data.input_features, cfg.data.target_features)
print(inputs.shape, targets.shape)
df.head()

In [ ]:
# 3. Split + window
splits = chronological_split(inputs, targets,
                              train_split=cfg.data.train_split,
                              val_split=cfg.data.val_split,
                              scaler_name=cfg.data.scaler)
X_tr, y_tr = sliding_window(splits.x_train, splits.y_train, cfg.window.look_back, cfg.window.horizon)
X_va, y_va = sliding_window(splits.x_val,   splits.y_val,   cfg.window.look_back, cfg.window.horizon)
X_te, y_te = sliding_window(splits.x_test,  splits.y_test,  cfg.window.look_back, cfg.window.horizon)
X_tr.shape, X_va.shape, X_te.shape

In [ ]:
# 4. Build & train
model = build_model(cfg.model, cfg.train,
                    look_back=cfg.window.look_back,
                    n_features_in=X_tr.shape[2],
                    n_outputs=y_tr.shape[1])
model.summary()
history = fit_model(model, X_tr, y_tr, X_va, y_va, cfg.train, verbose=2)

In [ ]:
# 5. Evaluate on test set in original units
y_pred_s = model.predict(X_te, verbose=0)
n_t = len(cfg.data.target_features)
y_pred = splits.y_scaler.inverse_transform(y_pred_s.reshape(-1, n_t))
y_true = splits.y_scaler.inverse_transform(y_te.reshape(-1, n_t))
compute_all(y_true, y_pred)

In [ ]:
# 6. Reconstruct the original CSV (fill the NaNs)
filled = reconstruct_series(model, inputs, targets,
                            splits.x_scaler, splits.y_scaler,
                            look_back=cfg.window.look_back,
                            horizon=cfg.window.horizon)
print('NaNs before:', targets.isna().sum().sum(),
      'after:', filled.isna().sum().sum())